In [1]:

# If you are on a fresh machine, uncomment the next block to install basics.
# --------------------------------------------------------------
# !pip install -U pip
# !pip install openai python-dotenv jupyter ipykernel tqdm pandas datasets pyarrow tiktoken requests pydantic matplotlib rich

import os, json, re, time, sys
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # loads OPENAI_API_KEY from .env if present

client = OpenAI()
print("Has OPENAI_API_KEY:", bool(os.getenv("OPENAI_API_KEY")))


MODEL_SMALL = "gpt-5-mini"  # you can switch to another model later

# Minimal single-shot call (no history, text in → text out)
resp = client.responses.create(
    model=MODEL_SMALL,
    input="what is 1 + 1?"
)
print(resp.output_text)


Has OPENAI_API_KEY: True
1 + 1 = 2.


In [ ]:
import os, csv, json
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
from sklearn.metrics import classification_report

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")

INPUT_FILE = "task2_cardib_cleared_assaulting_replies_added.csv"
PROMPTS_DIR = "prompts/"
OUTPUT_DIR = "outputs/"

VERSIONS = [
    "zero_shot", "one_shot", "three_shot",
    "five_shot", "cot", "five_shot_shuffled"
]

df = pd.read_csv(INPUT_FILE).dropna(subset=["text", "label"])  # 小样本跑

def run_prompt(version):
    prompt_template = open(f"{PROMPTS_DIR}{version}.txt").read()
    preds, golds = [], []
    for _, row in df.iterrows():
        prompt = prompt_template.format(text=row["text"])
        completion = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=50
        )
        output = completion.choices[0].message.content.strip().lower()
        if "positive" in output:
            preds.append("positive")
        elif "negative" in output:
            preds.append("negative")
        else:
            preds.append("neutral")
        golds.append(row["label"])

    os.makedirs(f"{OUTPUT_DIR}{version}", exist_ok=True)
    json.dump(
        {"preds": preds, "golds": golds},
        open(f"{OUTPUT_DIR}{version}/results.json", "w"), indent=2
    )

    report = classification_report(golds, preds, output_dict=True)
    pd.DataFrame(report).T.to_csv(f"{OUTPUT_DIR}{version}/metrics.csv")
    print(f"{version} done.")

for v in VERSIONS:
    run_prompt(v)

zero_shot done.
one_shot done.
three_shot done.
five_shot done.
cot done.
five_shot_shuffled done.


### Summary


In [9]:

summary = []
for v in VERSIONS:
    metrics_path = f"{OUTPUT_DIR}{v}/metrics.csv"
    if os.path.exists(metrics_path):
        df_metrics = pd.read_csv(metrics_path, index_col=0)
        if "accuracy" in df_metrics.index and "macro avg" in df_metrics.index:
            acc = df_metrics.loc["accuracy", "precision"]
            f1 = df_metrics.loc["macro avg", "f1-score"]
            summary.append({"version": v, "accuracy": acc, "macro_f1": f1})

if summary:
    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(f"{OUTPUT_DIR}/summary_results.csv", index=False)
    print("\nSummary written to outputs/summary_results.csv")
    print(summary_df)


Summary written to outputs/summary_results.csv
              version  accuracy  macro_f1
0           zero_shot     0.775  0.747980
1            one_shot     0.750  0.710943
2          three_shot     0.800  0.761886
3           five_shot     0.825  0.807265
4                 cot     0.475  0.467612
5  five_shot_shuffled     0.775  0.748504
